# 📓 Notebook 2 — FastAPI Backend Server
### Egyptian Telecom RAG Support Assistant

This notebook:
1. Loads the persisted ChromaDB vector store from Google Drive (created in Notebook 1)
2. Initializes an Ollama `llama3.2` LLM instance
3. Builds a RAG retrieval chain with a system prompt tailored to Egyptian telecom support
4. Exposes a FastAPI `/query` POST endpoint (with CORS enabled)
5. Tunnels port 8000 to a public URL via `localtunnel`

**Keep this notebook running** while you use Notebook 3 — it hosts the live API.

## Cell 1 — Install dependencies

In [ ]:
# Update apt's package index first, then install zstd (required by the Ollama installer).
# Skipping `apt-get update` is the usual reason this install silently fails on a fresh Colab VM.
!apt-get update -qq
!apt-get install -y zstd -qq

import shutil
if shutil.which("zstd") is None:
    raise RuntimeError(
        "zstd did not install correctly via apt-get. Try re-running this cell — "
        "if it still fails, run `!apt-get update && !apt-get install -y zstd` manually "
        "in a new cell and check the output for errors."
    )
print("✅ zstd installed at:", shutil.which("zstd"))

!pip install -q fastapi==0.115.5 uvicorn==0.32.1 pyngrok==7.2.1 nest_asyncio==1.6.0 \
    langchain==0.3.7 langchain-community==0.3.7 langchain-chroma==0.1.4 chromadb==0.5.20

# Install Node.js (required for localtunnel) and localtunnel itself
!curl -fsSL https://deb.nodesource.com/setup_18.x | sudo -E bash - > /dev/null 2>&1
!apt-get install -y nodejs > /dev/null 2>&1
!npm install -g localtunnel > /dev/null 2>&1

# Install Ollama (now that zstd is confirmed present)
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time

# Sanity check: confirm the ollama binary actually installed before trying to run it
ollama_path = shutil.which("ollama")
if ollama_path is None:
    raise RuntimeError(
        "Ollama installation failed — 'ollama' binary not found on PATH. "
        "Scroll up in this cell's output for the actual install error."
    )
print(f"✅ Ollama installed at: {ollama_path}")

subprocess.Popen([ollama_path, "serve"])
time.sleep(5)

# llama3.2 is used as the chat/generation LLM.
# nomic-embed-text is used for embeddings (must match Notebook 1's embedding model exactly,
# since llama3.2 does not support embedding requests on Ollama's backend).
!{ollama_path} pull llama3.2
!{ollama_path} pull nomic-embed-text

## Cell 2 — Mount Drive & load the persisted vector store

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = "/content/drive/MyDrive/TelecomRAG"
VECTOR_DIR = os.path.join(BASE_DIR, "chroma_db")

from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma

# Must match the embedding model used in Notebook 1 (nomic-embed-text), NOT llama3.2 —
# llama3.2 is only used for answer generation below, via a separate Ollama LLM instance.
embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma(
    persist_directory=VECTOR_DIR,
    embedding_function=embeddings,
    collection_name="telecom_support_kb"
)

print(f"✅ Loaded vector store with {vectorstore._collection.count()} vectors")

## Cell 3 — Initialize LLM and build the RAG chain

In [ ]:
from langchain_community.llms import Ollama
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA

llm = Ollama(model="llama3.2", temperature=0.2)

SYSTEM_PROMPT_TEMPLATE = """You are a helpful, professional customer support assistant for Egyptian telecom operators (WE, Vodafone, and Etisalat).

Rules you must follow:
- Answer ONLY using the information provided in the context below. If the context does not contain the answer, say you don't have enough information and suggest the customer contact their operator's official support line.
- Be concise, polite, and practical — customers want clear step-by-step help (e.g., router resets, mobile wallet activation, balance/quota checks, bundle activation).
- If the customer's question relates to a specific operator, tailor the answer to that operator's procedures found in the context.
- Never invent phone numbers, prices, or policies that are not in the context.
- Respond in the same language the customer used (Arabic or English).

Context:
{context}

Customer Question:
{question}

Helpful Answer:"""

RAG_PROMPT = PromptTemplate(
    template=SYSTEM_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)

def build_qa_chain(company_filter: str | None):
    """Builds a RetrievalQA chain, optionally filtered to a single operator."""
    search_kwargs = {"k": 4}
    if company_filter and company_filter != "All":
        search_kwargs["filter"] = {"company": company_filter}

    retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)

    chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
        chain_type_kwargs={"prompt": RAG_PROMPT}
    )
    return chain

print("✅ RAG chain builder ready")

## Cell 4 — Build the FastAPI application

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from typing import Literal, List

app = FastAPI(
    title="Egyptian Telecom RAG Support API",
    description="RAG-powered customer support backend for WE, Vodafone, and Etisalat",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class QueryRequest(BaseModel):
    question: str = Field(..., min_length=1, description="Customer's question")
    company: Literal["WE", "Vodafone", "Etisalat", "All"] = Field(
        default="All", description="Operator to filter knowledge base by"
    )

class SourceDocument(BaseModel):
    company: str
    source: str

class QueryResponse(BaseModel):
    answer: str
    company: str
    sources: List[SourceDocument]

@app.get("/")
def root():
    return {"status": "ok", "message": "Egyptian Telecom RAG Support API is running"}

@app.get("/health")
def health():
    try:
        count = vectorstore._collection.count()
        return {"status": "healthy", "vector_count": count}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/query", response_model=QueryResponse)
def query_endpoint(request: QueryRequest):
    try:
        chain = build_qa_chain(request.company)
        result = chain.invoke({"query": request.question})

        answer = result.get("result", "").strip()
        source_docs = result.get("source_documents", [])

        seen = set()
        unique_sources = []
        for doc in source_docs:
            company = doc.metadata.get("company", "Unknown")
            source = doc.metadata.get("source", "Unknown")
            key = (company, source)
            if key not in seen:
                seen.add(key)
                unique_sources.append(SourceDocument(company=company, source=source))

        return QueryResponse(
            answer=answer,
            company=request.company,
            sources=unique_sources
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error processing query: {str(e)}")

print("✅ FastAPI app defined")

## Cell 5 — Run the server in a background thread

In [ ]:
import nest_asyncio
import uvicorn
import threading
import time
import traceback
import requests
import asyncio

nest_asyncio.apply()

# Capture any startup exception from the background thread — by default, exceptions
# raised inside a thread are silently swallowed and never reach the notebook's output,
# which is why the server can "fail" with no visible error at all.
server_error = {"exception": None}

def run_server():
    try:
        # Force the standard asyncio event loop policy for this thread and build a
        # fresh loop manually, instead of relying on uvicorn.run()/nest_asyncio to sort
        # it out. A prior uvicorn run may have already installed uvloop as the
        # process-wide default policy — nest_asyncio can only patch a standard asyncio
        # loop, not uvloop's, which is why "Can't patch loop of type uvloop.Loop" kept
        # happening even after passing loop="asyncio" to uvicorn.run().
        asyncio.set_event_loop_policy(asyncio.DefaultEventLoopPolicy())
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

        config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info", loop="asyncio")
        server = uvicorn.Server(config)
        loop.run_until_complete(server.serve())
    except Exception as e:
        server_error["exception"] = e
        print("❌ Server thread crashed:")
        traceback.print_exc()

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Actively poll /health instead of blindly sleeping — this tells us definitively
# whether the server actually came up, rather than assuming a fixed delay was enough.
SERVER_READY = False
for attempt in range(20):
    time.sleep(1)
    if server_error["exception"] is not None:
        raise RuntimeError(
            "FastAPI server failed to start — see the traceback printed above from the "
            "background thread. Common causes: an error in an earlier cell (e.g. app, "
            "vectorstore, or llm not defined), or a bug in the /query endpoint code."
        )
    try:
        resp = requests.get("http://localhost:8000/health", timeout=2)
        if resp.status_code == 200:
            SERVER_READY = True
            break
    except requests.exceptions.ConnectionError:
        continue

if SERVER_READY:
    print("✅ FastAPI server started on port 8000 and responded to /health")
else:
    raise RuntimeError(
        "Server did not respond on port 8000 after 20 seconds, and no exception was "
        "captured either. Try re-running Cells 2–4 above (in order) to make sure "
        "`vectorstore`, `llm`, and `app` are all defined, then re-run this cell."
    )

## Cell 6 — Expose port 8000 via localtunnel

In [ ]:
import subprocess
import urllib.request

# Get the tunnel password (your Colab runtime's public IP)
public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf-8').strip()
print(f"🔑 Localtunnel password (if prompted): {public_ip}")

# Launch localtunnel pointing at port 8000
lt_process = subprocess.Popen(
    ["npx", "localtunnel", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

import time
time.sleep(4)

print("\n📡 Reading tunnel URL...")
for _ in range(20):
    line = lt_process.stdout.readline()
    if line:
        print(line.strip())
        if "your url is" in line.lower():
            break
    time.sleep(0.5)

print("\n⚠️ Copy the 'https://...loca.lt' URL above — paste it into Notebook 3's BACKEND_URL variable.")
print(f"⚠️ If localtunnel shows a password page in the browser, enter this IP: {public_ip}")

## Cell 7 — Quick local test

In [ ]:
import requests

test_response = requests.post(
    "http://localhost:8000/query",
    json={"question": "How do I reset my home router?", "company": "WE"}
)
print(test_response.status_code)
print(test_response.json())